<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat%2Fp4-hierarchical-prompt-tree/p4-hierarchical-prompt-tree/notebooks/p4_hierarchical_tree_colab_seed2021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Thực Nghiệm P4: TAPTA Hierarchical Prompt Tree & Decision Fusion
### Protocol v1 Track A (FewRel 8 tasks × 10 relations, strict 5-shot, Rehearsal-Free $M=0$, Seed 2021)

Notebook này thực thi phương pháp đề xuất **P4 (Hierarchical Prompt Tree & Decision-Level Soft Fusion)** trên môi trường **Google Colab GPU (CUDA)** mà **hoàn toàn không có Oracle Task ID lúc kiểm thử**:
- **Khắc phục nhược điểm của P3 (Prompt-Prototype Discrepancy):** Ở P3, hòa trộn tham số mồi nhắc ($P_{\text{soft}} = w_1 P_1 + w_2 P_2$) kéo vector kiểm thử sang không gian nội suy, làm lệch khỏi các nguyên mẫu quan hệ tĩnh $p_c$ được tính lúc huấn luyện. P4 chuyển sang **Tổ hợp mềm ở Tầng Quyết định (*Decision-Level Soft Fusion*)**:
  $$\mathcal{S}_{\text{fusion}}(x, c) = \sum_{k \in \mathcal{T}_K(x)} w_k(x) \cdot \cos\left(h_r(x; [P^{\text{root}}; P^{\text{leaf}}_k]), p_c\right)$$
- **Cấu trúc Cây Mồi nhắc 2 Tầng (Hierarchical Prompt Tree Decomposition):**
  - Tầng Gốc ($P^{\text{root}}, L_{\text{root}}=4$): Chia sẻ chung cho mọi tác vụ, neo giữ không gian ngữ cảnh quan hệ bất biến, đóng băng sau Task 0 để chống trôi tri thức.
  - Tầng Nhánh ($P^{\text{leaf}}_t, L_{\text{leaf}}=4$): Độc lập theo từng tác vụ, tối ưu hóa kèm hàm mất mát trực giao $\mathcal{L}_{\text{orth}}$ để phân tách biểu diễn quan hệ.
  - Tổng chiều dài: $L = 4 + 4 = 8$ tokens (bảo toàn ngân sách tính toán).
- **Bảng đối chuẩn mục tiêu:**
  - P0 (Shared Prompt — Sàn): `52.83%`
  - P2b (Learned Router): `56.04%` (Router Acc: `50.85%`)
  - P3 (Soft Top-K Param Blend): `55.50%` (Top-2 Coverage: `81.41%`, Top-1 Gated: `59.25%`)
  - P2a (Centroid Hard Router): `59.35%` (Router Acc: `64.69%`)
  - P1 (Oracle-Task Prompt — Trần lý tưởng): `70.14%`
  - **Mục tiêu P4:** Vượt qua mốc `59.35%` của P2a, kỳ vọng đạt **`61.5% - 64.0%`**, thu hẹp $\ge 50\%$ khoảng cách tới trần Oracle P1.

## 1. Kiểm tra Môi trường GPU (NVIDIA CUDA Preflight Check)

In [1]:
import torch
print('=' * 60)
print('CUDA / GPU PREFLIGHT CHECK')
print('=' * 60)
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device:     {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    raise RuntimeError('Vui lòng bật GPU trên Google Colab: Runtime -> Change runtime type -> T4 GPU!')

CUDA / GPU PREFLIGHT CHECK
CUDA Available: True
GPU Device:     Tesla T4
Total VRAM:     14.56 GB


## 2. Clone / Cập nhật Repository `research-code-lab` (Nhánh `feat/p4-hierarchical-prompt-tree`)

In [2]:
import os
REPO_URL = 'https://github.com/vinhlq2512/research-code-lab.git'
BRANCH = 'feat/p4-hierarchical-prompt-tree'
REPO_DIR = '/content/research-code-lab'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd /content/research-code-lab
!pip install -q transformers pyyaml

Cloning into '/content/research-code-lab'...
remote: Enumerating objects: 3156, done.
remote: Counting objects: 100% (3156/3156), done.
remote: Compressing objects: 100% (2239/2239), done.
remote: Total 3156 (delta 888), reused 2967 (delta 716), pack-reused 0 (from 0)
Receiving objects: 100% (3156/3156), 18.09 MiB | 18.25 MiB/s, done.
Resolving deltas: 100% (888/888), done.
/content/research-code-lab


## 3. Chạy Thực Nghiệm P4: Hierarchical Prompt Tree & Decision Fusion

In [3]:
!python experiments/run_p4.py \
    --config configs/p4_hierarchical_prompt_tree_fewrel_5shot_seed2021.yaml \
    --device cuda

Continual Relation Extraction: P4 (p4_hierarchical_tree)
Description: Proposed Method P4 (TAPTA Hierarchical Tree): Frozen BERT with 2-tier Hierarchical Prompt Tree (L_root=4, L_leaf=4), Sub-Centroid Task Routing, and Decision-Level Soft Prototype Fusion (Zero Oracle Task ID)
Prompt Tree: L_root=4 | L_leaf=4 | L_total=8
Decision Fusion: K=2 | Temp=0.2
Seed:        2021
Results Dir: /content/research-code-lab/runs/p4_hierarchical_tree/fewrel_5shot/seed_2021

[1/4] Loaded Protocol v1 Dataset: 8 continual tasks.
[2026-11-24 03:36:59,788] [INFO] HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
[2026-11-24 03:37:00,020] [INFO] HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"
[2026-11-24 03:37:00,257] [INFO] HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"
[2026-11-24 03:37:00,504] [INFO] HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/86

## 4. Kiểm tra Kết quả & Đối sánh Toàn diện (P0 $\to$ P4 $\to$ P1)

In [4]:
import json
from pathlib import Path

results_path = Path('runs/p4_hierarchical_tree/fewrel_5shot/seed_2021/summary.json')
if results_path.exists():
    with results_path.open() as f:
        p4_summary = json.load(f)

    print('=' * 85)
    print('BẢNG ĐỐI SÁNH HIỆU NĂNG TOÀN DIỆN (PROTOCOL V1 TRACK A FEWREL 5-SHOT)')
    print('=' * 85)
    print(f'{"Mô hình":<32} | {"Proto / Fusion Acc":<20} | {"Router Acc":<12} | {"Top-K Cov":<10} | {"Gap Closed":<10}')
    print('-' * 85)
    print(f'{"P0: Shared Prompt (Sàn)":<32} | 52.83%               | N/A          | N/A        | 0.00%')
    print(f'{"P2b: Learned MLP Router":<32} | 56.04%               | 50.85%       | N/A        | 18.56%')
    print(f'{"P3: Soft Top-K Router (Global)":<32} | 55.50%               | 64.69%       | 81.41%     | 15.45%')
    print(f'{"P3: Soft Top-K Router (Gated)":<32} | 59.25%               | 64.69%       | 81.41%     | 37.11%')
    print(f'{"P2a: Centroid Cosine Router":<32} | 59.35%               | 64.69%       | N/A        | 37.69%')

    p4_fusion = p4_summary.get('final_decision_fusion_accuracy', 0.0) * 100
    p4_fusion_gated = p4_summary.get('final_decision_fusion_topk_gated_accuracy', 0.0) * 100
    p4_tree_top1 = p4_summary.get('final_tree_top1_prototype_accuracy', 0.0) * 100
    p4_tree_top1_gated = p4_summary.get('final_tree_top1_gated_prototype_accuracy', 0.0) * 100
    p4_r1 = p4_summary.get('final_router_top1_accuracy', 0.0) * 100
    p4_rk = p4_summary.get('final_router_topk_coverage', 0.0) * 100
    p4_gap = p4_summary.get('oracle_gap_closed_pct', 0.0)

    print(f'{"P4: Decision Fusion (Primary)":<32} | {p4_fusion:5.2f}%               | {p4_r1:5.2f}%       | {p4_rk:5.2f}%    | {p4_gap:5.2f}%')
    print(f'{"P4: Decision Fusion (Top-K Gated)":<32} | {p4_fusion_gated:5.2f}%               | -            | -          | -')
    print(f'{"P4: Tree Top-1 Proto":<32} | {p4_tree_top1:5.2f}%               | -            | -          | -')
    print(f'{"P4: Tree Top-1 Gated Proto":<32} | {p4_tree_top1_gated:5.2f}%               | -            | -          | -')
    print(f'{"P1: Oracle-Task (Trần)":<32} | 70.14%               | 100.00%      | 100.00%    | 100.00%')
    print('=' * 85)
else:
    print('Chưa tìm thấy file summary.json. Vui lòng chạy cell 3 trước.')

BẢNG ĐỐI SÁNH HIỆU NĂNG TOÀN DIỆN (PROTOCOL V1 TRACK A FEWREL 5-SHOT)
Mô hình                          | Proto / Fusion Acc   | Router Acc   | Top-K Cov  | Gap Closed
-------------------------------------------------------------------------------------
P0: Shared Prompt (Sàn)          | 52.83%               | N/A          | N/A        | 0.00%
P2b: Learned MLP Router          | 56.04%               | 50.85%       | N/A        | 18.56%
P3: Soft Top-K Router (Global)   | 55.50%               | 64.69%       | 81.41%     | 15.45%
P3: Soft Top-K Router (Gated)    | 59.25%               | 64.69%       | 81.41%     | 37.11%
P2a: Centroid Cosine Router      | 59.35%               | 64.69%       | N/A        | 37.69%
P4: Decision Fusion (Primary)    | 57.57%               | 64.69%       | 81.41%    | 34.51%
P4: Decision Fusion (Top-K Gated) | 57.93%               | -            | -          | -
P4: Tree Top-1 Proto             | 58.38%               | -            | -          | -
P4: Tree Top-1

## 5. Đóng gói & Tải về Kết quả (Download Results Artifact)

In [5]:
import shutil
from google.colab import files

archive_dir = 'runs/p4_hierarchical_tree/fewrel_5shot/seed_2021'
zip_name = '/content/p4_hierarchical_tree_results_seed2021'

shutil.make_archive(zip_name, 'zip', archive_dir)
print(f'Đã đóng gói thành công: {zip_name}.zip')

# Tự động tải về máy
files.download(f'{zip_name}.zip')

Đã đóng gói thành công: /content/p4_hierarchical_tree_results_seed2021.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>